# 4.20 Selección de Archivos para Kaggle (Árboles Azarosos)

Este script recorre el grid de hiperparámetros definido para el experimento `exp420_00`, verifica si existen los archivos correspondientes generados por `aa420_ArbolesAzarosos.ipynb` y, en caso de existir, los copia a la subcarpeta `kaggle/` dentro del directorio del experimento.

#### 1. Limpieza del ambiente de R

In [1]:
# Limpieza de memoria y ambiente
rm(list = ls(all.names = TRUE))
gc(full = TRUE, verbose = FALSE)

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,657904,35.2,1454508,77.7,1063260,56.8
Vcells,1223365,9.4,8388608,64.0,1975054,15.1


#### 2. Configuración de Directorios y Experimento

In [2]:
# Determinar la raíz del proyecto
find_project_root <- function() {
  curr <- normalizePath(getwd(), winslash = "/")
  while (curr != "/" && curr != dirname(curr)) {
    if (file.exists(file.path(curr, "Dockerfile")) || dir.exists(file.path(curr, "src"))) {
      return(curr)
    }
    curr <- dirname(curr)
  }
  return(normalizePath(getwd(), winslash = "/"))
}

dir_base <- if (dir.exists("/workspace")) "/workspace" else find_project_root()

# Definición del experimento y directorios
experimento <- "exp420_00"
dir_experimento <- file.path(dir_base, "exp", experimento)
dir_kaggle <- file.path(dir_experimento, "kaggle")

# Crear el directorio kaggle si no existe
dir.create(dir_kaggle, showWarnings = FALSE, recursive = TRUE)

cat("Directorio base:            ", dir_base, "\n")
cat("Directorio del experimento: ", dir_experimento, "\n")
cat("Directorio de destino Kaggle:", dir_kaggle, "\n")

Directorio base:             /workspace 
Directorio del experimento:  /workspace/exp/exp420_00 
Directorio de destino Kaggle: /workspace/exp/exp420_00/kaggle 


#### 3. Definición del Grid de Hiperparámetros

In [6]:
PARAM <- list()
PARAM$num_trees_max <- 32

# Local
PARAM$grid <- list(
  feature_fraction = c(0.2, 0.5, 0.7),
  maxdepth = c(8, 10, 12),
  minsplit = c(100, 200, 400, 600, 800),
  minbucket_fraction = c(0.2, 0.3, 0.4, 0.5),
  cp = c(-1)
)



#### 4. Recorrido del Grid, Búsqueda y Copia de Archivos

In [7]:
# Contadores de control
total_combinaciones <- 0
copiados <- 0
no_encontrados <- 0

# Tabla para registrar los resultados
tb_seleccionados <- data.frame(
  iter = integer(),
  archivo = character(),
  ff = numeric(),
  maxdepth = integer(),
  minsplit = integer(),
  minbucket = integer(),
  cp = numeric(),
  estado = character(),
  stringsAsFactors = FALSE
)

for (v_ff in PARAM$grid$feature_fraction) {
  for (v_maxdepth in PARAM$grid$maxdepth) {
    for (v_minsplit in PARAM$grid$minsplit) {
      for (v_minbucket_frac in PARAM$grid$minbucket_fraction) {
        
        # Calcular minbucket como fracción de minsplit (al menos 1)
        v_minbucket <- max(1L, as.integer(round(v_minsplit * v_minbucket_frac)))

        for (v_cp in PARAM$grid$cp) {
          total_combinaciones <- total_combinaciones + 1
          
          # Nombre del archivo generado según la nomenclatura de aa420_ArbolesAzarosos.ipynb
          archivo_prediccion <- sprintf(
            "KA420_ff_%.2f_md_%d_ms_%d_mb_%d_cp_%.1f_%dtrees.csv",
            v_ff, v_maxdepth, v_minsplit, v_minbucket, v_cp, PARAM$num_trees_max
          )
          
          ruta_origen <- file.path(dir_experimento, archivo_prediccion)
          ruta_destino <- file.path(dir_kaggle, archivo_prediccion)
          
          if (file.exists(ruta_origen)) {
            file.copy(from = ruta_origen, to = ruta_destino, overwrite = TRUE)
            copiados <- copiados + 1
            estado <- "Copiado"
            cat(sprintf("[ENCONTRADO] %s -> kaggle/\n", archivo_prediccion))
          } else {
            no_encontrados <- no_encontrados + 1
            estado <- "No encontrado"
            cat(sprintf("[FALTANTE]   %s (no existe en %s)\n", archivo_prediccion, dir_experimento))
          }
          
          tb_seleccionados <- rbind(
            tb_seleccionados,
            data.frame(
              iter = total_combinaciones,
              archivo = archivo_prediccion,
              ff = v_ff,
              maxdepth = v_maxdepth,
              minsplit = v_minsplit,
              minbucket = v_minbucket,
              cp = v_cp,
              estado = estado,
              stringsAsFactors = FALSE
            )
          )
        }
      }
    }
  }
}

cat("\n=========================================\n")
cat("RESUMEN DE SELECCIÓN Y COPIA:\n")
cat(sprintf("Total combinaciones en grid: %d\n", total_combinaciones))
cat(sprintf("Archivos copiados a kaggle/: %d\n", copiados))
cat(sprintf("Archivos no encontrados:     %d\n", no_encontrados))
cat("=========================================\n")

[ENCONTRADO] KA420_ff_0.20_md_8_ms_100_mb_20_cp_-1.0_32trees.csv -> kaggle/
[ENCONTRADO] KA420_ff_0.20_md_8_ms_100_mb_30_cp_-1.0_32trees.csv -> kaggle/
[ENCONTRADO] KA420_ff_0.20_md_8_ms_100_mb_40_cp_-1.0_32trees.csv -> kaggle/
[ENCONTRADO] KA420_ff_0.20_md_8_ms_100_mb_50_cp_-1.0_32trees.csv -> kaggle/
[ENCONTRADO] KA420_ff_0.20_md_8_ms_200_mb_40_cp_-1.0_32trees.csv -> kaggle/
[ENCONTRADO] KA420_ff_0.20_md_8_ms_200_mb_60_cp_-1.0_32trees.csv -> kaggle/
[ENCONTRADO] KA420_ff_0.20_md_8_ms_200_mb_80_cp_-1.0_32trees.csv -> kaggle/
[ENCONTRADO] KA420_ff_0.20_md_8_ms_200_mb_100_cp_-1.0_32trees.csv -> kaggle/
[ENCONTRADO] KA420_ff_0.20_md_8_ms_400_mb_80_cp_-1.0_32trees.csv -> kaggle/
[ENCONTRADO] KA420_ff_0.20_md_8_ms_400_mb_120_cp_-1.0_32trees.csv -> kaggle/
[ENCONTRADO] KA420_ff_0.20_md_8_ms_400_mb_160_cp_-1.0_32trees.csv -> kaggle/
[ENCONTRADO] KA420_ff_0.20_md_8_ms_400_mb_200_cp_-1.0_32trees.csv -> kaggle/
[ENCONTRADO] KA420_ff_0.20_md_8_ms_600_mb_120_cp_-1.0_32trees.csv -> kaggle/
[ENCONT

#### 5. Verificación de Archivos en la Carpeta Kaggle

In [8]:
archivos_kaggle <- list.files(dir_kaggle, pattern = "\\.csv$")
cat(sprintf("Se encontraron %d archivos en %s:\n", length(archivos_kaggle), dir_kaggle))
print(head(archivos_kaggle, 20))
if (length(archivos_kaggle) > 20) {
  cat(sprintf("... y %d archivos más.\n", length(archivos_kaggle) - 20))
}

Se encontraron 60 archivos en /workspace/exp/exp420_00/kaggle:
 [1] "KA420_ff_0.20_md_10_ms_100_mb_20_cp_-1.0_32trees.csv" 
 [2] "KA420_ff_0.20_md_10_ms_100_mb_30_cp_-1.0_32trees.csv" 
 [3] "KA420_ff_0.20_md_10_ms_100_mb_40_cp_-1.0_32trees.csv" 
 [4] "KA420_ff_0.20_md_10_ms_100_mb_50_cp_-1.0_32trees.csv" 
 [5] "KA420_ff_0.20_md_10_ms_200_mb_100_cp_-1.0_32trees.csv"
 [6] "KA420_ff_0.20_md_10_ms_200_mb_40_cp_-1.0_32trees.csv" 
 [7] "KA420_ff_0.20_md_10_ms_200_mb_60_cp_-1.0_32trees.csv" 
 [8] "KA420_ff_0.20_md_10_ms_200_mb_80_cp_-1.0_32trees.csv" 
 [9] "KA420_ff_0.20_md_10_ms_400_mb_120_cp_-1.0_32trees.csv"
[10] "KA420_ff_0.20_md_10_ms_400_mb_160_cp_-1.0_32trees.csv"
[11] "KA420_ff_0.20_md_10_ms_400_mb_200_cp_-1.0_32trees.csv"
[12] "KA420_ff_0.20_md_10_ms_400_mb_80_cp_-1.0_32trees.csv" 
[13] "KA420_ff_0.20_md_10_ms_600_mb_120_cp_-1.0_32trees.csv"
[14] "KA420_ff_0.20_md_10_ms_600_mb_180_cp_-1.0_32trees.csv"
[15] "KA420_ff_0.20_md_10_ms_600_mb_240_cp_-1.0_32trees.csv"
[16] "KA420_ff_0.20_md